# 05 Second Purchase Propensity Model

This notebook builds a supervised ML model to predict whether a first-time buyer will make a second purchase within 60 days. The model supports CRM prioritisation for the second purchase accelerator strategy.

Inputs:

- `../data/processed/clean_transactions.parquet`

Outputs:

- `../data/processed/second_purchase_model_metrics.csv`
- `../data/processed/second_purchase_propensity_scores.csv`
- `../data/processed/second_purchase_propensity_lift.csv`
- `../data/processed/second_purchase_propensity_bands.csv`
- `../data/processed/second_purchase_feature_importance.csv`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROCESSED_DIR = Path("../data/processed")
TRANSACTIONS_PATH = PROCESSED_DIR / "clean_transactions.parquet"

if not TRANSACTIONS_PATH.exists():
    raise FileNotFoundError(
        f"Required input not found: {TRANSACTIONS_PATH}. Run notebook 01 first."
    )

RANDOM_STATE = 42
TARGET_WINDOW_DAYS = 60

## Build First-Order Modelling Dataset

The model uses only information available at first purchase: first-order value, basket size, product diversity, product-description keyword signals, country, and purchase timing. The target is whether the customer places a second valid order within 60 days.

In [ ]:
transactions = pd.read_parquet(TRANSACTIONS_PATH)
transactions.columns = (
    transactions.columns.str.strip().str.lower().str.replace(" ", "_")
)
transactions = transactions.rename(
    columns={
        "invoiceno": "invoice_no",
        "stockcode": "stock_code",
        "invoicedate": "invoice_date",
        "unitprice": "unit_price",
        "customerid": "customer_id",
    }
)
transactions["invoice_date"] = pd.to_datetime(transactions["invoice_date"])
valid_lines = transactions[transactions["is_valid_order_line"]].copy()
valid_lines["description_clean"] = valid_lines["description"].fillna("").astype(str).str.lower()

customer_orders = (
    valid_lines.groupby(["customer_id", "invoice_no"], as_index=False)
    .agg(
        order_date=("invoice_date", "min"),
        order_revenue=("analytical_revenue", "sum"),
        order_quantity=("quantity", "sum"),
        order_lines=("stock_code", "size"),
        unique_products=("stock_code", "nunique"),
        avg_unit_price=("unit_price", "mean"),
        max_unit_price=("unit_price", "max"),
        description_text=("description_clean", lambda values: " ".join(values.head(80))),
        country=("country", lambda values: values.mode().iloc[0] if not values.mode().empty else values.iloc[0]),
    )
)
customer_orders = customer_orders[customer_orders["order_revenue"] > 0].copy()
customer_orders = customer_orders.sort_values(["customer_id", "order_date", "invoice_no"]).reset_index(drop=True)
customer_orders["order_rank"] = customer_orders.groupby("customer_id").cumcount() + 1

first_orders = customer_orders[customer_orders["order_rank"] == 1].copy()
second_orders = customer_orders[customer_orders["order_rank"] == 2][["customer_id", "order_date"]].rename(columns={"order_date": "second_order_date"})
model_data = first_orders.merge(second_orders, on="customer_id", how="left")
model_data["days_to_second_purchase"] = (model_data["second_order_date"] - model_data["order_date"]).dt.days
model_data["repeat_within_60_days"] = model_data["days_to_second_purchase"].le(TARGET_WINDOW_DAYS).fillna(False).astype(int)

analysis_end_date = customer_orders["order_date"].max()
model_data["days_observed_after_first_purchase"] = (analysis_end_date - model_data["order_date"]).dt.days
model_data = model_data[model_data["days_observed_after_first_purchase"] >= TARGET_WINDOW_DAYS].copy()

model_data["first_order_month"] = model_data["order_date"].dt.month.astype(str)
model_data["first_order_weekday"] = model_data["order_date"].dt.day_name()
model_data["first_order_hour"] = model_data["order_date"].dt.hour
model_data["first_order_above_median_revenue"] = (
    model_data["order_revenue"] > model_data["order_revenue"].median()
).astype(int)
model_data["basket_diversity_ratio"] = model_data["unique_products"] / model_data["order_lines"]
model_data["revenue_per_line"] = model_data["order_revenue"] / model_data["order_lines"]
model_data["revenue_per_unique_product"] = model_data["order_revenue"] / model_data["unique_products"]
model_data["is_uk_customer"] = model_data["country"].eq("United Kingdom").astype(int)
model_data["contains_gift_keyword"] = model_data["description_text"].str.contains("gift", regex=False).astype(int)
model_data["contains_set_keyword"] = model_data["description_text"].str.contains("set", regex=False).astype(int)
model_data["contains_bag_keyword"] = model_data["description_text"].str.contains("bag", regex=False).astype(int)
model_data["contains_christmas_keyword"] = model_data["description_text"].str.contains("christmas", regex=False).astype(int)
model_data["contains_jumbo_keyword"] = model_data["description_text"].str.contains("jumbo", regex=False).astype(int)
model_data["log_first_order_revenue"] = np.log1p(model_data["order_revenue"])
model_data["log_order_quantity"] = np.log1p(model_data["order_quantity"].clip(lower=0))
model_data["log_avg_unit_price"] = np.log1p(model_data["avg_unit_price"].clip(lower=0))
model_data["log_revenue_per_line"] = np.log1p(model_data["revenue_per_line"].clip(lower=0))

print(f"Model rows: {len(model_data):,}")
print(f"Target rate: {model_data['repeat_within_60_days'].mean():.2%}")
model_data.head()

## Time-Based Train/Test Split

The split is based on first purchase date rather than random sampling. This better reflects a real scoring workflow, where the model is trained on older customers and evaluated on later customers.

In [ ]:
feature_columns = [
    "log_first_order_revenue",
    "log_order_quantity",
    "order_lines",
    "unique_products",
    "basket_diversity_ratio",
    "log_avg_unit_price",
    "log_revenue_per_line",
    "revenue_per_unique_product",
    "first_order_hour",
    "first_order_above_median_revenue",
    "is_uk_customer",
    "contains_gift_keyword",
    "contains_set_keyword",
    "contains_bag_keyword",
    "contains_christmas_keyword",
    "contains_jumbo_keyword",
    "country",
    "first_order_month",
    "first_order_weekday",
]
target_column = "repeat_within_60_days"

split_date = model_data["order_date"].quantile(0.8)
train_data = model_data[model_data["order_date"] <= split_date].copy()
test_data = model_data[model_data["order_date"] > split_date].copy()

X_train = train_data[feature_columns]
y_train = train_data[target_column]
X_test = test_data[feature_columns]
y_test = test_data[target_column]

print(f"Split date: {split_date.date()}")
print(f"Train rows: {len(train_data):,}, target rate: {y_train.mean():.2%}")
print(f"Test rows: {len(test_data):,}, target rate: {y_test.mean():.2%}")

## Train Baseline and ML Models

The notebook trains an interpretable logistic regression baseline, a random forest model, and a histogram gradient boosting model. The selected CRM scoring model prioritises precision in the top 20% score band because the commercial use case is campaign prioritisation rather than broad population classification.

In [ ]:
numeric_features = [
    "log_first_order_revenue",
    "log_order_quantity",
    "order_lines",
    "unique_products",
    "basket_diversity_ratio",
    "log_avg_unit_price",
    "log_revenue_per_line",
    "revenue_per_unique_product",
    "first_order_hour",
    "first_order_above_median_revenue",
    "is_uk_customer",
    "contains_gift_keyword",
    "contains_set_keyword",
    "contains_bag_keyword",
    "contains_christmas_keyword",
    "contains_jumbo_keyword",
]
categorical_features = ["country", "first_order_month", "first_order_weekday"]

numeric_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocess = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocess, numeric_features),
        ("categorical", categorical_preprocess, categorical_features),
    ]
)

models = {
    "logistic_regression": Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ]
    ),
    "random_forest": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    min_samples_leaf=20,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "hist_gradient_boosting": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                HistGradientBoostingClassifier(
                    learning_rate=0.05,
                    max_iter=200,
                    min_samples_leaf=30,
                    l2_regularization=0.1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

def precision_at_fraction(y_true, probabilities, fraction):
    cutoff = max(1, int(np.ceil(len(y_true) * fraction)))
    top_index = np.argsort(probabilities)[::-1][:cutoff]
    return y_true.iloc[top_index].mean()

model_results = []
fitted_models = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[model_name] = model
    test_probability = model.predict_proba(X_test)[:, 1]
    test_prediction = (test_probability >= 0.5).astype(int)
    model_results.append(
        {
            "model": model_name,
            "train_rows": len(train_data),
            "test_rows": len(test_data),
            "test_target_rate": y_test.mean(),
            "roc_auc": roc_auc_score(y_test, test_probability),
            "pr_auc": average_precision_score(y_test, test_probability),
            "brier_score": brier_score_loss(y_test, test_probability),
            "accuracy_at_50pct_threshold": accuracy_score(y_test, test_prediction),
            "precision_at_50pct_threshold": precision_score(y_test, test_prediction, zero_division=0),
            "recall_at_50pct_threshold": recall_score(y_test, test_prediction, zero_division=0),
            "precision_top_10pct": precision_at_fraction(y_test, test_probability, 0.10),
            "precision_top_20pct": precision_at_fraction(y_test, test_probability, 0.20),
            "precision_top_30pct": precision_at_fraction(y_test, test_probability, 0.30),
        }
    )

model_metrics = pd.DataFrame(model_results).sort_values(
    ["precision_top_20pct", "roc_auc"],
    ascending=False,
).reset_index(drop=True)
selected_model_name = model_metrics.loc[0, "model"]
selected_model = fitted_models[selected_model_name]
model_metrics.to_csv(PROCESSED_DIR / "second_purchase_model_metrics.csv", index=False)

print(f"Selected model: {selected_model_name}")
model_metrics

## Score Customers and Build Lift Table

Customers are scored using the selected model. The lift table ranks test customers by propensity decile to show whether high-scored groups have higher observed repeat rates.

In [ ]:
model_data["second_purchase_propensity_score"] = selected_model.predict_proba(model_data[feature_columns])[:, 1]
model_data["data_split"] = np.where(model_data["order_date"] <= split_date, "train", "test")
model_data["propensity_band"] = pd.cut(
    model_data["second_purchase_propensity_score"],
    bins=[-np.inf, 0.40, 0.55, np.inf],
    labels=["Low propensity", "Medium propensity", "High propensity"],
)

test_scored = model_data[model_data["data_split"] == "test"].copy()
test_scored = test_scored.sort_values("second_purchase_propensity_score", ascending=False).reset_index(drop=True)
test_scored["score_decile"] = pd.qcut(
    test_scored.index + 1,
    10,
    labels=[
        "Top 10%",
        "10-20%",
        "20-30%",
        "30-40%",
        "40-50%",
        "50-60%",
        "60-70%",
        "70-80%",
        "80-90%",
        "Bottom 10%",
    ],
)

overall_test_rate = test_scored[target_column].mean()
lift = (
    test_scored.groupby("score_decile", observed=False)
    .agg(
        customers=("customer_id", "count"),
        observed_repeat_rate=(target_column, "mean"),
        avg_score=("second_purchase_propensity_score", "mean"),
    )
    .reset_index()
)
lift["lift_vs_test_average"] = lift["observed_repeat_rate"] / overall_test_rate
lift.to_csv(PROCESSED_DIR / "second_purchase_propensity_lift.csv", index=False)

propensity_band_summary = (
    model_data.groupby(["data_split", "propensity_band"], observed=False)
    .agg(
        customers=("customer_id", "count"),
        observed_repeat_rate=(target_column, "mean"),
        avg_score=("second_purchase_propensity_score", "mean"),
    )
    .reset_index()
)
propensity_band_summary.to_csv(PROCESSED_DIR / "second_purchase_propensity_bands.csv", index=False)

score_columns = [
    "customer_id",
    "order_date",
    "country",
    "order_revenue",
    "order_quantity",
    "order_lines",
    "unique_products",
    "repeat_within_60_days",
    "days_to_second_purchase",
    "data_split",
    "second_purchase_propensity_score",
    "propensity_band",
]
model_data[score_columns].to_csv(PROCESSED_DIR / "second_purchase_propensity_scores.csv", index=False)

lift

## Feature Importance

Feature importance is directional. It should be used to understand model behaviour, not as proof of causal impact.

In [ ]:
model_step = selected_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    feature_names = selected_model.named_steps["preprocess"].get_feature_names_out()
    importance_values = model_step.feature_importances_
    feature_importance = pd.DataFrame(
        {
            "feature": feature_names,
            "importance": importance_values,
        }
    )
elif hasattr(model_step, "coef_"):
    feature_names = selected_model.named_steps["preprocess"].get_feature_names_out()
    importance_values = np.abs(model_step.coef_[0])
    feature_importance = pd.DataFrame(
        {
            "feature": feature_names,
            "importance": importance_values,
        }
    )
else:
    permutation = permutation_importance(
        selected_model,
        X_test,
        y_test,
        n_repeats=10,
        random_state=RANDOM_STATE,
        scoring="roc_auc",
    )
    feature_importance = pd.DataFrame(
        {
            "feature": feature_columns,
            "importance": permutation.importances_mean,
        }
    )

feature_importance = feature_importance.sort_values("importance", ascending=False)
feature_importance.to_csv(PROCESSED_DIR / "second_purchase_feature_importance.csv", index=False)
feature_importance.head(20)

In [ ]:
expected_outputs = {
    "second_purchase_model_metrics.csv": {"model", "roc_auc", "pr_auc", "brier_score", "precision_top_20pct"},
    "second_purchase_propensity_scores.csv": {"customer_id", "second_purchase_propensity_score", "propensity_band", "data_split"},
    "second_purchase_propensity_lift.csv": {"score_decile", "observed_repeat_rate", "lift_vs_test_average"},
    "second_purchase_propensity_bands.csv": {"data_split", "propensity_band", "observed_repeat_rate"},
    "second_purchase_feature_importance.csv": {"feature", "importance"},
}
for file_name, expected_columns in expected_outputs.items():
    path = PROCESSED_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f"Expected output was not created: {path}")
    output = pd.read_csv(path)
    missing_columns = sorted(expected_columns - set(output.columns))
    if missing_columns:
        raise ValueError(f"{file_name} is missing columns: {missing_columns}")
    print(f"Validated {file_name}: {output.shape}")